## Tips

- **Start simple** — a single GRU with 64 units is a good first attempt.
  Only add complexity (stacking, dropout, bidirectional) if the simple model
  is clearly underfitting or overfitting.

- **Use many epochs** — RNNs on time series converge slowly. 10 or 20 epochs
  is almost never enough. Use **at least 100 epochs** and let early stopping
  decide when to stop. The best checkpoint is saved automatically by
  `ModelCheckpoint` — you will not miss the optimal point even if you
  train for too long.

- **Watch the train/val gap** — if train MAE is much lower than val MAE
  you are overfitting. Add dropout, reduce `hidden_size`, or increase
  early stopping patience to give regularization more time to work.

- **Use TensorBoard** — run `tensorboard --logdir runs` in your terminal
  to monitor train and val curves in real time. A healthy training curve
  shows both losses decreasing together. A diverging val curve means overfitting.

- **Learning rate matters** — if the loss is not decreasing after the first
  few epochs, try a lower learning rate (`1e-4` instead of `1e-3`).
  Use `ReduceLROnPlateau` to automatically decay the learning rate when
  val MAE stops improving.

- **The sklearn baseline is hard to beat** — `HistGradientBoosting` with
  explicit lag features is a very strong baseline for tabular time series.
  This is not a failure of the RNN — it reflects a fundamental difference
  between the two approaches: tree models get temporal information from
  hand-crafted features, while RNNs must learn it from raw sequences.
  Getting within 10 bikes/hour of the sklearn baseline (< 44 bikes/hour)
  is an excellent result.

In [1]:
import os
import numpy as np
import joblib

from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn

from torch.utils.tensorboard import SummaryWriter

In [2]:
save_dir = "data/bike_processed"

# --- Load arrays ---
raw_data    = np.load(os.path.join(save_dir, "raw_data.npy"))
counts = np.load(os.path.join(save_dir, "counts.npy"))
#count_stats = np.load(os.path.join(save_dir, "count_stats.npy"))
train_idx   = np.load(os.path.join(save_dir, "train_idx.npy"))
val_idx     = np.load(os.path.join(save_dir, "val_idx.npy"))
test_idx    = np.load(os.path.join(save_dir, "test_idx.npy"))
naive_mae   = np.load(os.path.join(save_dir, "naive_mae.npy"))
preprocessor = joblib.load(os.path.join(save_dir, "preprocessor_rnn.pkl"))

# --- Recover split sizes ---
num_train = len(train_idx)
num_val   = len(val_idx)
num_test  = len(test_idx)

count_mean, count_std = np.mean(counts[train_idx]), np.std(counts[train_idx])
# --- Count stats in the training set ---
#count_mean  = count_stats[0]
#count_std   = count_stats[1]

# --- Sanity check ---
print(f"raw_data shape:    {raw_data.shape}")
print(f"counts shape: {counts.shape}")
print(f"train/val/test:    {num_train} / {num_val} / {num_test}")
print(f"counts range: {counts.min():.0f} to {counts.max():.0f} bikes/hour")
print(f"\nNaive baseline — Val MAE: {naive_mae[0]:.2f} | Test MAE: {naive_mae[1]:.2f} bikes/hour")

raw_data shape:    (17210, 28)
counts shape: (17210,)
train/val/test:    12047 / 2581 / 2582
counts range: 1 to 977 bikes/hour

Naive baseline — Val MAE: 93.26 | Test MAE: 80.78 bikes/hour


In [3]:
count_norm = (counts - count_mean) / count_std

In [4]:
class TimeseriesDataset(Dataset):
    def __init__(self, data, targets, sequence_length, sampling_rate, 
                 start_index, end_index, shuffle=False):
        self.data            = data
        self.targets         = targets
        self.sequence_length = sequence_length
        self.sampling_rate   = sampling_rate

        # Valid starting indices: each sequence of length sequence_length
        # sampled every sampling_rate steps needs:
        # (sequence_length - 1) * sampling_rate + 1 rows ahead
        self.indices = np.arange(start_index, end_index)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start = self.indices[idx]
        # Sample every `sampling_rate` steps for `sequence_length` steps
        steps = np.arange(start, start + self.sequence_length * self.sampling_rate, 
                          self.sampling_rate)
        x = self.data[steps]
        # Target is `delay` steps ahead of the sequence start
        y = self.targets[start + delay]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

In [5]:
sampling_rate   = 1    # already hourly
sequence_length = 24   # look back 24 hours
delay           = 1    # predict 1 hour ahead
batch_size      = 256

train_dataset = TimeseriesDataset(
    data=raw_data, targets=count_norm,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=0, end_index=num_train - (sequence_length - 1) * sampling_rate - delay,
)
val_dataset = TimeseriesDataset(
    data=raw_data, targets=count_norm,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train, end_index=num_train + num_val - (sequence_length - 1) * sampling_rate - delay,
)
test_dataset = TimeseriesDataset(
    data=raw_data, targets=count_norm,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train + num_val, end_index=len(raw_data) - (sequence_length - 1) * sampling_rate - delay,
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

# --- Sanity check ---
inputs, targets = next(iter(train_loader))
print(f"Input shape:  {inputs.shape}")   # (256, 24, num_features)
print(f"Target shape: {targets.shape}")  # (256,)
print(f"Target range: {targets.min():.0f} to {targets.max():.0f}")

Input shape:  torch.Size([256, 24, 28])
Target shape: torch.Size([256])
Target range: -1 to 3


In [6]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss = 0.0
    total_mae  = 0.0
    n          = 0
    with torch.set_grad_enabled(training):
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            preds = model(inputs)
            loss  = criterion(preds, targets)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * inputs.size(0)
            total_mae  += torch.sum(torch.abs(preds - targets)).item()
            n          += inputs.size(0)
    mae = (total_mae / n) * count_std
    return total_loss / n, mae 

def get_predictions(model, dataset):
    model.eval()
    all_preds     = []
    all_targets   = []
    loader = DataLoader(dataset, batch_size=256, shuffle=False)
    with torch.no_grad():
        for inputs, targets in loader:
            preds = model(inputs.to(device)).cpu().numpy()
            all_preds.append(preds * count_std + count_mean)  # un-normalize predictions
            all_targets.append(targets.numpy() * count_std + count_mean)  # un-normalize targets
    return np.concatenate(all_preds), np.concatenate(all_targets)


In [7]:
class ModelCheckpoint:
    """Saves the best model based on a monitored metric."""
    def __init__(self, filepath, monitor="val_mae", mode="min", verbose=True):
        self.filepath = filepath
        self.monitor  = monitor
        self.verbose  = verbose
        self.best     = float("inf") if mode == "min" else float("-inf")
        self.mode     = mode

    def step(self, metrics, model=None):
        value    = metrics[self.monitor]
        improved = value < self.best if self.mode == "min" else value > self.best
        if improved:
            self.best = value
            torch.save(model.state_dict(), self.filepath)
            if self.verbose:
                print(f"  ✓ Best model saved ({self.monitor}: {value:.2f} bikes)")
        return improved

class EarlyStopping:
    """Stops training when a monitored metric stops improving."""
    def __init__(self, monitor="val_mae", patience=5, min_delta=1e-4, mode="min"):
        self.monitor     = monitor
        self.patience    = patience
        self.min_delta   = min_delta
        self.mode        = mode
        self.best        = float("inf") if mode == "min" else float("-inf")
        self.counter     = 0
        self.should_stop = False

    def step(self, metrics, model=None):
        value    = metrics[self.monitor]
        improved = (value < self.best - self.min_delta if self.mode == "min"
                    else value > self.best + self.min_delta)
        if improved:
            self.best    = value
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
                print(f"  Early stopping triggered (no improvement for {self.patience} epochs)")
        return improved

class ReduceLROnPlateau:
    """Wraps PyTorch scheduler with the same callback interface."""
    def __init__(self, optimizer, monitor="val_mae", patience=3, factor=0.5):
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, patience=patience, factor=factor
        )
        self.monitor = monitor

    def step(self, metrics, model=None):
        self.scheduler.step(metrics[self.monitor])

In [8]:
class GRUModel(nn.Module):
    def __init__(self, num_features, hidden_size=64, dropout=0.2):
        super().__init__()
        self.gru  = nn.GRU(input_size=num_features, hidden_size=hidden_size,
                           batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # out: (batch, seq_len, hidden_size) — take last timestep
        out, _ = self.gru(x)
        return self.head(self.drop(out[:, -1, :])).squeeze(-1)

In [9]:
# --- Setup ---
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = GRUModel(num_features=raw_data.shape[-1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()
writer    = SummaryWriter(log_dir="runs/bike_gru")

callbacks = [
    ModelCheckpoint("bike_gru_best.pt", monitor="val_mae"),
    EarlyStopping(monitor="val_mae", patience=10),
    ReduceLROnPlateau(optimizer, monitor="val_mae", patience=5, factor=0.5),
]

# --- Training loop ---
epochs = 100
for epoch in range(1, epochs + 1):
    train_loss, train_mae = run_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_mae   = run_epoch(model, val_loader,   criterion)

    metrics = {"train_loss": train_loss, "train_mae": train_mae,
               "val_loss":   val_loss,   "val_mae":   val_mae}

    writer.add_scalars("Loss", {"train": train_loss, "val": val_loss}, epoch)
    writer.add_scalars("MAE",  {"train": train_mae,  "val": val_mae},  epoch)

    print(f"Epoch {epoch:02d} — "
          f"train loss: {train_loss:.4f}, train MAE: {train_mae:.2f} | "
          f"val loss: {val_loss:.4f}, val MAE: {val_mae:.2f}")

    for cb in callbacks:
        cb.step(metrics, model)

    if any(isinstance(cb, EarlyStopping) and cb.should_stop for cb in callbacks):
        print(f"  Training stopped at epoch {epoch}.")
        break

writer.close()

Epoch 01 — train loss: 0.6915, train MAE: 96.64 | val loss: 1.2453, val MAE: 131.12
  ✓ Best model saved (val_mae: 131.12 bikes)
Epoch 02 — train loss: 0.4106, train MAE: 69.33 | val loss: 0.9766, val MAE: 110.39
  ✓ Best model saved (val_mae: 110.39 bikes)
Epoch 03 — train loss: 0.3698, train MAE: 66.26 | val loss: 0.9159, val MAE: 105.10
  ✓ Best model saved (val_mae: 105.10 bikes)
Epoch 04 — train loss: 0.3472, train MAE: 63.96 | val loss: 0.9084, val MAE: 107.54
Epoch 05 — train loss: 0.3151, train MAE: 60.79 | val loss: 0.7419, val MAE: 93.19
  ✓ Best model saved (val_mae: 93.19 bikes)
Epoch 06 — train loss: 0.2633, train MAE: 54.64 | val loss: 0.5736, val MAE: 79.01
  ✓ Best model saved (val_mae: 79.01 bikes)
Epoch 07 — train loss: 0.1902, train MAE: 46.45 | val loss: 0.4372, val MAE: 69.23
  ✓ Best model saved (val_mae: 69.23 bikes)
Epoch 08 — train loss: 0.1434, train MAE: 41.00 | val loss: 0.3322, val MAE: 60.10
  ✓ Best model saved (val_mae: 60.10 bikes)
Epoch 09 — train loss

In [10]:
# --- Reload best and evaluate ---
model.load_state_dict(torch.load("bike_gru_best.pt", map_location=device))
_, test_mae = run_epoch(model, test_loader, criterion)
print(f"\nTest MAE:     {test_mae:.2f} bikes/hour")
print(f"Naive MAE:    {naive_mae[1]:.2f} bikes/hour")
print(f"Gap to naive: {test_mae - naive_mae[1]:.2f} bikes/hour")


Test MAE:     37.76 bikes/hour
Naive MAE:    80.78 bikes/hour
Gap to naive: -43.02 bikes/hour
